In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
CSV_PATH = "raw_laptops.csv"   # <-- update if using Google Drive path

df_raw = pd.read_csv(CSV_PATH)
print("Shape:", df_raw.shape)
df_raw.head()


In [ ]:
# ------------------------------------------------------------
# CELL 3 - Data cleaning & verification
# ------------------------------------------------------------
print("Duplicate model_name rows:", df_raw.duplicated(subset=["model_name"]).sum())

print("\nMissing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

df = df_raw.copy()
df = df.drop_duplicates(subset=["model_name"]).reset_index(drop=True)

# --- Regex Extraction for Clean Numerical Features ---
df["price_inr"] = df["raw_price"].astype(str).str.replace("₹", "").str.replace(",", "").str.strip()
df["price_inr"] = pd.to_numeric(df["price_inr"], errors="coerce")

df["ram_gb"] = df["raw_ram"].astype(str).str.extract(r'(\d+)').astype(float)
df["storage_gb"] = df["raw_storage"].astype(str).str.extract(r'(\d+)').astype(float)
df["weight_kg"] = df["raw_weight"].astype(str).str.extract(r'([\d\.]+)').astype(float)

res_split = df["raw_resolution"].astype(str).str.extract(r'(\d+)\s*x\s*(\d+)')
df["res_width"] = pd.to_numeric(res_split[0], errors="coerce")
df["res_height"] = pd.to_numeric(res_split[1], errors="coerce")

df["screen_size_in"] = df["raw_display_size"].astype(str).str.extract(r'([\d\.]+)').astype(float)

# --- Structural missingness treatment ---
df["ram_gb"] = df["ram_gb"].fillna(df["ram_gb"].median())
df["storage_gb"] = df["storage_gb"].fillna(df["storage_gb"].median())
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())
df["screen_size_in"] = df["screen_size_in"].fillna(df["screen_size_in"].median())
df["res_width"] = df["res_width"].fillna(1920)
df["res_height"] = df["res_height"].fillna(1080)
df["os"] = df["os"].fillna("Windows 11")
df["touchscreen"] = df["touchscreen"].fillna("No")

# Drop rows missing target variable (price_inr)
df = df.dropna(subset=["price_inr"]).reset_index(drop=True)

print("\nMissing values after treatment:")
remaining = df[["price_inr", "ram_gb", "storage_gb", "weight_kg", "res_width", "res_height"]].isna().sum()
print(remaining[remaining > 0] if len(remaining[remaining > 0]) else "None")

# Business-rule consistency checks
invalid_price = (df["price_inr"] <= 0).sum()
invalid_ram = (df["ram_gb"] <= 0).sum()
print(f"\nInvalid price values: {invalid_price}")
print(f"Invalid RAM values:   {invalid_ram}")


In [ ]:
# --- Feature Engineering ---
df["price_per_ram"] = df["price_inr"] / (df["ram_gb"] + 1e-5)
df["price_per_storage"] = df["price_inr"] / (df["storage_gb"] + 1e-5)
df["total_pixels"] = df["res_width"] * df["res_height"]
df["ppi"] = (np.sqrt(df["res_width"]**2 + df["res_height"]**2)) / df["screen_size_in"]

df["is_high_ram"] = (df["ram_gb"] >= 16).astype(int)
df["is_heavy"] = (df["weight_kg"] >= 2.0).astype(int)

print("Rows remaining:", len(df))
df[['price_inr', 'ram_gb', 'storage_gb', 'total_pixels', 'ppi', 'price_per_ram']].head()


In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df["price_inr"], bins=40, kde=True)
plt.title("Distribution of Laptop Prices (Target)")
plt.xlabel("Price in INR (₹)")
plt.show()

print(df["price_inr"].describe())


In [ ]:
brand_avg = df.groupby("brand")["price_inr"].mean().sort_values()

plt.figure(figsize=(10, 4))
brand_avg.plot(kind="barh")
plt.title("Average Laptop Price by Brand")
plt.xlabel("Mean Price (₹)")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby("brand")["price_inr"].count().sort_values().plot(
    kind="barh", ax=axes[0], title="Total Laptops Count by Brand")

df.groupby("touchscreen")["price_inr"].mean().plot(
    kind="bar", ax=axes[1], title="Avg Price by Touchscreen Support")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x="ram_gb", y="price_inr", data=df)
plt.title("Price Distribution across RAM Capacities")
plt.xlabel("RAM (GB)")
plt.ylabel("Price (₹)")
plt.show()

plt.figure(figsize=(6, 4))
sns.scatterplot(x="weight_kg", y="price_inr", data=df, alpha=0.6)
plt.title("Weight vs Laptop Price")
plt.xlabel("Weight (kg)")
plt.ylabel("Price (₹)")
plt.show()


In [ ]:
numeric_cols = [
    "ram_gb", "storage_gb", "weight_kg", "screen_size_in",
    "res_width", "res_height", "total_pixels", "ppi",
    "price_per_ram", "price_per_storage", "price_inr"
]

plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap (Laptop Features)")
plt.show()


In [ ]:
# ------------------------------------------------------------
# CELL 10 - One-Hot Encoding (pd.get_dummies)
# ------------------------------------------------------------
categorical_cols = ["brand", "os", "touchscreen", "backlit_keyboard"]

# One-Hot Encoding converts text categories into binary 0/1 columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, dtype=int)

print("Shape after One-Hot Encoding:", df_encoded.shape)
df_encoded.head()


In [ ]:
# ------------------------------------------------------------
# CELL 11 - Feature Scaling & Train-Test Split
# ------------------------------------------------------------
exclude_cols = [
    "laptop_id", "model_name", "raw_price", "raw_processor", "raw_cores",
    "raw_ram", "ram_bus_speed", "raw_storage", "raw_display_size",
    "raw_resolution", "refresh_rate", "raw_gpu", "raw_weight",
    "raw_dimensions", "warranty", "utility", "price_inr"
]

feature_cols = [c for c in df_encoded.columns if c not in exclude_cols]

X = df_encoded[feature_cols]
y = df_encoded["price_inr"]

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling using StandardScaler (Mean=0, Std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Normalization using MinMaxScaler (0 to 1 range)
minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)

print("✅ Feature Scaling & Dataset Partitioning Complete!")
print("X shape:", X.shape)
print("X_train_scaled shape (Standardized):", X_train_scaled.shape)
print("X_test_scaled shape (Standardized): ", X_test_scaled.shape)
print("\ny summary (Target Price):\n", y.describe())


In [ ]:
# ------------------------------------------------------------
# CELL 12 - Save Cleaned Dataset
# ------------------------------------------------------------
OUTPUT_CLEAN_CSV = "cleaned_laptops.csv"

df_encoded.to_csv(OUTPUT_CLEAN_CSV, index=False)

print(f"Cleaned dataset successfully saved to {OUTPUT_CLEAN_CSV}")
print("Final Cleaned Data Shape:", df_encoded.shape)
df_encoded.head()


In [ ]:
# ------------------------------------------------------------
# CELL 13 - Supervised Model Training & Evaluation
# ------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import joblib

# 1. Baseline Median
med_val = y_train.median()
y_pred_base = np.full_like(y_test, med_val)
base_mae = mean_absolute_error(y_test, y_pred_base)
base_r2 = r2_score(y_test, y_pred_base)

# 2. Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, np.log1p(y_train))
y_pred_ridge = np.expm1(ridge.predict(X_test_scaled))
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

# 3. HistGradientBoosting
hgb = HistGradientBoostingRegressor(max_iter=150, random_state=42)
hgb.fit(X_train_scaled, np.log1p(y_train))
y_pred_hgb = np.expm1(hgb.predict(X_test_scaled))
hgb_mae = mean_absolute_error(y_test, y_pred_hgb)
hgb_r2 = r2_score(y_test, y_pred_hgb)

# 4. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=200, max_depth=16, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, np.log1p(y_train))
y_pred_rf = np.expm1(rf.predict(X_test_scaled))
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = root_mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print('=== SUPERVISED REGRESSION BENCHMARK ===')
print(f'Baseline Median:      MAE = INR {base_mae:,.0f} | R2 = {base_r2:.3f}')
print(f'Ridge Regression:     MAE = INR {ridge_mae:,.0f} | R2 = {ridge_r2:.3f}')
print(f'HistGradientBoosting: MAE = INR {hgb_mae:,.0f} | R2 = {hgb_r2:.3f}')
print(f'Random Forest:        MAE = INR {rf_mae:,.0f} | RMSE = INR {rf_rmse:,.0f} | R2 = {rf_r2:.3f}')


In [ ]:
# ------------------------------------------------------------
# CELL 14 - Actual vs Predicted & Feature Importances
# ------------------------------------------------------------
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_rf, alpha=0.6, color='#0f172a')
max_val = max(y_test.max(), y_pred_rf.max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect Fit')
plt.xlabel('Actual Catalog Price (₹)')
plt.ylabel('Predicted Price (₹)')
plt.title(f'Random Forest: Actual vs Predicted (R² = {rf_r2:.3f})')
plt.legend()

plt.subplot(1, 2, 2)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
importances.plot(kind='barh', color='#334155')
plt.title('Top 10 Feature Importances')
plt.xlabel('Gini Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# CELL 15 - Save Trained Model Artifact
# ------------------------------------------------------------
import os
os.makedirs('../../models', exist_ok=True)
joblib.dump(rf, '../../models/laptop_price_model_standalone.joblib')
print('Trained model successfully saved to models/laptop_price_model_standalone.joblib')
